# Calculating hosting capacity for SDG&E
Use the hosting capacity calculation methodology from Brockway et al (2021) to calculate the hosting capacity for SDG&E at the census tract level. 

In [1]:
import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import box
from shapely.geometry import MultiLineString
import matplotlib.pyplot as plt

### Load in data

In [14]:
# read in feederline data
sdge_feederlines = gpd.read_file("../../../../capstone/electrigrid/data/utilities/sandiego_grid/gen_cap_lines_sdge.geojson")

# read in multifamily data
sdge_multifamily = gpd.read_file("../../../../capstone/electrigrid/data/building_zillow_merges/sdcounty_buildings/multi_summed_units_sd.geojson")

# read in the single family data  
sdge_singlefamily = gpd.read_file("../../../../capstone/electrigrid/data/building_zillow_merges/sdcounty_buildings/sdge_singlefamily.geojson")

# read in the census tract data
census_tracts = gpd.read_file("../../../../capstone/electrigrid/data/census/tl_2025_06_tract/tl_2025_06_tract.shp")

# Link the homes to the nearest feederline

In [10]:

# change the crs to a projected CRS
sdge_feederlines = sdge_feederlines.to_crs("EPSG:3310")
sdge_singlefamily = sdge_singlefamily.to_crs("EPSG:3310")

# add a check
assert sdge_feederlines.crs == sdge_singlefamily.crs


# index the data
sdge_feederlines.sindex
sdge_singlefamily.sindex

# spatial join
sdge_singlefamily_linked = gpd.sjoin_nearest(sdge_singlefamily, 
                                        sdge_feederlines, 
                                        how="left", 
                                        lsuffix='_left', 
                                        rsuffix='_right',
                                        distance_col='dist_to_line_m')

In [11]:
# change the crs to a projected CRS
sdge_multifamily = sdge_multifamily.to_crs("EPSG:3310")

# add a check
assert sdge_feederlines.crs == sdge_multifamily.crs

# index the data
sdge_feederlines.sindex
sdge_multifamily.sindex

# spatial join
sdge_multifamily_linked = gpd.sjoin_nearest(sdge_multifamily, 
                                        sdge_feederlines, 
                                        how="left", 
                                        lsuffix='_left', 
                                        rsuffix='_right',
                                        distance_col='dist_to_line_m')

# Create line length column

Our data analysis will need length data for each line segment. Let's calculate the lengths.

In [ ]:
# create length column in metres
sdge_singlefamily_linked['LENGTH (M)'] = sdge_singlefamily_linked.length
sdge_multifamily_linked['LENGTH (M)'] = sdge_multifamily_linked.length

In [13]:
sdge_singlefamily_linked.head()

,type,year,room,heat,cool,own,unit,value,sqft_type,sqft,...,OHUG,LABELTEXT,ICAWNOF_PVGENERATION,LABELTEXT_ICA,RESTRICTED,ICAWOF_UNIGENERATION_LC,ICAWOF_UNILOAD_LC,ICAWNOF_UNIGENERATION_LC,dist_to_line_m,LENGTH (M)
0,Single,1947.0,3.0,None,None,O,1.0,117939.0,living,1456.0,...,OH,None,1.0,Up to 1.00,N,ICA_Operation_Flex,Load_Thermal,ICA_Voltage_Delta,244.699737,0.0
1,Single,1958.0,2.0,None,None,None,1.0,44966.0,living,576.0,...,OH,None,1.0,Up to 1.00,N,ICA_Operation_Flex,Load_Thermal,ICA_Voltage_Delta,241.795582,0.0
2,Single,2000.0,1.0,None,None,None,1.0,173672.0,living,2400.0,...,OH,None,1.0,Up to 1.00,N,ICA_Operation_Flex,Load_Thermal,ICA_Voltage_Delta,176.875959,0.0
3,Single,1948.0,2.0,None,None,None,1.0,85328.0,living,1024.0,...,OH,None,1.0,Up to 1.00,N,ICA_Operation_Flex,Load_Thermal,ICA_Voltage_Delta,162.951749,0.0
4,Single,2002.0,2.0,None,None,O,1.0,371922.0,living,2145.0,...,OH,None,0.9,Up to 1.00,N,ICA_Operation_Flex,Load_Thermal,ICA_Voltage_Delta,12.242524,0.0


# Map all of the data sources